In [ ]:
import pandas as pd
from pathlib import Path

# Directory path
path = Path(
    "/Volumes/T7/Li Lab/Projects/HEU/OLD/2025-12-14_HEU_Metabolomics_V3/heu_data/12_03_2025_empirical_compounds/"
)

# List all files (equivalent to list.files(..., full.names = TRUE))
files = [f for f in path.glob("*.csv") if not f.name.startswith("._")]

print(files)


[PosixPath('/Volumes/T7/Li Lab/Projects/HEU/OLD/2025-12-14_HEU_Metabolomics_V3/heu_data/12_03_2025_empirical_compounds/RP_neg_empirical_compounds.csv'), PosixPath('/Volumes/T7/Li Lab/Projects/HEU/OLD/2025-12-14_HEU_Metabolomics_V3/heu_data/12_03_2025_empirical_compounds/RP_pos_empirical_compounds.csv'), PosixPath('/Volumes/T7/Li Lab/Projects/HEU/OLD/2025-12-14_HEU_Metabolomics_V3/heu_data/12_03_2025_empirical_compounds/HILIC_neg_empirical_compounds.csv'), PosixPath('/Volumes/T7/Li Lab/Projects/HEU/OLD/2025-12-14_HEU_Metabolomics_V3/heu_data/12_03_2025_empirical_compounds/HILIC_pos_empirical_compounds.csv')]


In [9]:
# Read and combine (map_dfr equivalent)
df_list = []
for f in files:
    temp = pd.read_csv(f, sep=None, engine="python")  # auto-detect delimiter
    temp["source_file"] = f.name
    df_list.append(temp)

all_khipu_files = pd.concat(df_list, ignore_index=True)


# Create 'mode' column (case_when equivalent)
def assign_mode(fname):
    if "HILIC_neg" in fname:
        return "HILIC_neg"
    elif "HILIC_pos" in fname:
        return "HILIC_pos"
    elif "RP_neg" in fname:
        return "RP_neg"
    elif "RP_pos" in fname:
        return "RP_pos"
    else:
        return None


all_khipu_files["mode"] = all_khipu_files["source_file"].apply(assign_mode)

all_khipu_files.head()

,id,mz,rtime,peak_area,detection_counts,isotope,modification,ion_relation,kp_id,source_file,mode
0,F174,70.0117,3.00,2.085710e+09,591.0,13C/12C,M-H-,"13C/12C,M-H-",kp1_70.0156,RP_neg_empirical_compounds.csv,RP_neg
1,F362,110.0349,4.57,1.606874e+09,572.0,M0,ACN,"M0,ACN",kp1_70.0156,RP_neg_empirical_compounds.csv,RP_neg
2,F131,69.0083,3.00,1.804139e+09,600.0,M0,M-H-,"M0,M-H-",kp1_70.0156,RP_neg_empirical_compounds.csv,RP_neg
3,F9695,134.0543,0.77,1.565818e+06,1.0,13C/12C,ACN,"13C/12C,ACN",kp2_93.0345,RP_neg_empirical_compounds.csv,RP_neg
4,F9510,133.0509,0.63,5.272237e+08,174.0,M0,ACN,"M0,ACN",kp2_93.0345,RP_neg_empirical_compounds.csv,RP_neg


## L4 annotations to HMDB and LMSD

In [ ]:
from jms.dbStructures import ExperimentalEcpdDatabase, knownCompoundDatabase
import json


def l4_annotate(mode, all_khipu_files, annotation_sources, rt_tolerance=5):
    mode_df = all_khipu_files[all_khipu_files["mode"] == mode]
    mode_df = mode_df[mode_df["kp_id"].str.startswith("kp")]
    dict_empcpds = {}
    for kp_id, group in mode_df.groupby("kp_id"):
        neutral_formula_mass = float(kp_id.split("_", 1)[1])
        features = group.drop(columns=["source_file", "mode"]).to_dict("records")
        for f in features:
            f["id_number"] = f["id"]
        dict_empcpds[kp_id] = {
            "interim_id": kp_id,
            "neutral_formula_mass": neutral_formula_mass,
            "MS1_pseudo_Spectra": features,
        }

    jms_mode = "neg" if "neg" in mode else "pos"
    EED = ExperimentalEcpdDatabase(mode=jms_mode, rt_tolerance=rt_tolerance)
    EED.build_from_list_empCpds(list(dict_empcpds.values()))

    formula_entry_lookup = {}
    for source in annotation_sources:
        with open(source, encoding="utf-8") as fh:
            source_data = json.load(fh)
        print(f"Loaded {len(source_data)} entries from {source}")
        for entry in source_data:
            formula_entry_lookup.setdefault(entry["neutral_formula"], []).append(entry)
        KCD = knownCompoundDatabase()
        KCD.mass_index_list_compounds(source_data)
        KCD.build_emp_cpds_index()
        EED.extend_empCpd_annotation(KCD)

    khipus_with_l4 = 0
    total_l4 = 0
    for khipu in dict_empcpds.values():
        khipu.setdefault("Level_4", [])
        for match in khipu.get("list_matches", []):
            formula_mass, _, _ = match
            formula, _ = formula_mass.split("_")
            if formula in formula_entry_lookup:
                khipu["Level_4"].extend(formula_entry_lookup[formula])
        if khipu["Level_4"]:
            khipus_with_l4 += 1
            total_l4 += len(khipu["Level_4"])

    print(
        f"L4: {khipus_with_l4}/{len(dict_empcpds)} khipus annotated, {total_l4} total entries"
    )

    return dict_empcpds


def get_l4_matches(dict_empcpds):
    annotation_table = []
    kp2ft = khipu_id_to_feature_id(dict_empcpds)

    for kp_id, khipu in dict_empcpds.items():
        for feature in kp2ft[kp_id]:
            l4_annots = khipu.get("Level_4", [])
            for l4_annot in l4_annots:
                l4_annot_entry = {}
                l4_annot_entry.update({"feature": feature, "level": "4"})
                l4_annot_entry.update(l4_annot)
                annotation_table.append(l4_annot_entry)

    return pd.DataFrame(annotation_table)


def khipu_id_to_feature_id(dict_empcpds):
    """
    Returns a mapping from khipu id to the feature ids contained in that khipu.

    Returns:
        dict: kp_id to list of feature ids
    """
    khipu_id_to_feature_id = {}

    for kp_id, khipu in dict_empcpds.items():
        khipu_id_to_feature_id[kp_id] = []
        for peak in khipu["MS1_pseudo_Spectra"]:
            khipu_id_to_feature_id[kp_id].append(peak["id_number"])

    return khipu_id_to_feature_id

In [29]:
jm_annot_dir = "/Users/chongj/Desktop/Li_Lab/annotation_sources/JM_annotation_sources/annotation_sources/"
l4_libs = ["hmdb_metabolites.json", "LMSD.json"]

l4_full_paths = [jm_annot_dir + f for f in l4_libs]

l4_annotations_list = {}

modes = ["HILIC_neg", "HILIC_pos", "RP_neg", "RP_pos"]

for m in modes:
    l4_annotations = l4_annotate(
        mode=m,
        all_khipu_files=all_khipu_files,
        annotation_sources=l4_full_paths,
        rt_tolerance=5,
    )

    df = get_l4_matches(l4_annotations)
    df["mode"] = m
    l4_annotations_list[m] = df

mega_l4_annotations = pd.concat(l4_annotations_list.values(), ignore_index=True)
mega_l4_annotations.head()

Loaded 217899 entries from /Users/chongj/Desktop/Li_Lab/annotation_sources/JM_annotation_sources/annotation_sources/hmdb_metabolites.json
Loaded 47390 entries from /Users/chongj/Desktop/Li_Lab/annotation_sources/JM_annotation_sources/annotation_sources/LMSD.json
L4: 4203/17976 khipus annotated, 32448 total entries
Loaded 217899 entries from /Users/chongj/Desktop/Li_Lab/annotation_sources/JM_annotation_sources/annotation_sources/hmdb_metabolites.json
Loaded 47390 entries from /Users/chongj/Desktop/Li_Lab/annotation_sources/JM_annotation_sources/annotation_sources/LMSD.json
L4: 4694/18748 khipus annotated, 35628 total entries
Loaded 217899 entries from /Users/chongj/Desktop/Li_Lab/annotation_sources/JM_annotation_sources/annotation_sources/hmdb_metabolites.json
Loaded 47390 entries from /Users/chongj/Desktop/Li_Lab/annotation_sources/JM_annotation_sources/annotation_sources/LMSD.json
L4: 1598/6569 khipus annotated, 16571 total entries
Loaded 217899 entries from /Users/chongj/Desktop/Li_L

,feature,level,accession,primary_id,name,chemical_formula,neutral_formula,neutral_formula_mass,monisotopic_molecular_weight,iupac_name,...,cas_registry_number,smiles,inchi,inchikey,primary_db,SMILES,other_ids,category,class,mode
0,F55040,4,HMDB0028750,HMDB0028750,Aspartyl-Cysteine,C7H12N2O5S,C7H12N2O5S,236.046693,236.046692194,3-amino-3-[(1-carboxy-2-sulfanylethyl)carbamoy...,...,,NC(CC(O)=O)C(=O)NC(CS)C(O)=O,InChI=1S/C7H12N2O5S/c8-3(1-5(10)11)6(12)9-4(2-...,FKBFDTRILNZGAI-UHFFFAOYSA-N,HMDBv5,NaN,NaN,NaN,NaN,HILIC_neg
1,F55040,4,HMDB0028771,HMDB0028771,Cysteinyl-Aspartate,C7H12N2O5S,C7H12N2O5S,236.046693,236.046692194,2-(2-amino-3-sulfanylpropanamido)butanedioic acid,...,,NC(CS)C(=O)NC(CC(O)=O)C(O)=O,InChI=1S/C7H12N2O5S/c8-3(2-15)6(12)9-4(7(13)14...,TULNGKSILXCZQT-UHFFFAOYSA-N,HMDBv5,NaN,NaN,NaN,NaN,HILIC_neg
2,F55040,4,HMDB0244752,HMDB0244752,N-Acetyl-3-(nitrosulfanyl)-L-valine,C7H12N2O5S,C7H12N2O5S,236.046693,236.046692668,2-acetamido-3-methyl-3-(nitrosulfanyl)butanoic...,...,,CC(=O)NC(C(O)=O)C(C)(C)S[N+]([O-])=O,"InChI=1S/C7H12N2O5S/c1-4(10)8-5(6(11)12)7(2,3)...",GTMKBVNAJMQYKF-UHFFFAOYSA-N,HMDBv5,NaN,NaN,NaN,NaN,HILIC_neg
3,F55040,4,HMDB0250498,HMDB0250498,Coumestan,C15H8O3,C15H8O3,236.047344,236.047344118,"8,17-dioxatetracyclo[8.7.0.0^{2,7}.0^{11,16}]h...",...,,O=C1OC2=CC=CC=C2C2=C1C1=CC=CC=C1O2,InChI=1S/C15H8O3/c16-15-13-9-5-1-3-7-11(9)17-1...,JBIZUYWOIKFETJ-UHFFFAOYSA-N,HMDBv5,NaN,NaN,NaN,NaN,HILIC_neg
4,F39963,4,HMDB0028750,HMDB0028750,Aspartyl-Cysteine,C7H12N2O5S,C7H12N2O5S,236.046693,236.046692194,3-amino-3-[(1-carboxy-2-sulfanylethyl)carbamoy...,...,,NC(CC(O)=O)C(=O)NC(CS)C(O)=O,InChI=1S/C7H12N2O5S/c8-3(1-5(10)11)6(12)9-4(2-...,FKBFDTRILNZGAI-UHFFFAOYSA-N,HMDBv5,NaN,NaN,NaN,NaN,HILIC_neg


In [31]:
import os
from datetime import date

ms1_output_dir = "/Users/chongj/Desktop/Li_Lab/Projects/HEU/2026-05-21_MS1/"
os.makedirs(ms1_output_dir, exist_ok=True)

csv_path = os.path.join(
    ms1_output_dir,
    f"{date.today().strftime('%Y-%m-%d')}_heu_combined_all_ms1_l4_annotations.csv",
)
mega_l4_annotations.to_csv(csv_path, index=False)
print(f"Saved CSV  ({len(mega_l4_annotations)} rows)    → {csv_path}")

Saved CSV  (429149 rows)    → /Users/chongj/Desktop/Li_Lab/Projects/HEU/2026-05-21_MS1/2026-05-28_heu_combined_all_ms1_l4_annotations.csv


## CSM Annotations

In [42]:
import os
import sys
import json

csm_path = "/Users/chongj/Desktop/Li_Lab/github/consensus_serum_metabolome-main/"
sys.path.append(csm_path + "utils")
from csm_align import *
from csm_annotate import *

In [43]:
# DB_CPDS_SERUM	DB_REF_CSM

CSM_PATH = csm_path + "r1_libs_1.6.2/"
db_cpds_serum = CSM_PATH + "blood_metabolites_20250212.json"
neu_csm = CSM_PATH + "r1_neu_mass_registries_annotated_1012.json"

mDict, neuCSM, csmlib_dict = load_csm_full(
    db_cpds_serum,
    neu_csm,
    lib_hilicpos=CSM_PATH + "r1_ref_hilic_pos_csmfs_20250304.json",
    lib_rppos=CSM_PATH + "r1_ref_rp_pos_csmfs_20250304.json",
    lib_hilicneg=CSM_PATH + "r1_ref_hilic_neg_csmfs_20250304.json",
    lib_rpneg=CSM_PATH + "r1_ref_rp_neg_csmfs_20250304.json",
)

# Mass of common chemical formulas
KCD_formula_coordinate = build_KCD_from_formula_coordinate(formula_coordinate)

In [46]:
csmlib_dict.keys()

dict_keys(['hilicpos', 'rppos', 'hilicneg', 'rpneg'])

In [55]:
ft_tbl_path = "/Volumes/T7/Li Lab/Projects/HEU/OLD/2025-12-14_HEU_Metabolomics_V3/heu_data/12_03_2025_final_feature_tables/"

ft_tables = [f for f in os.listdir(ft_tbl_path) if not f.startswith("._")]
print(ft_tables)

output_dir = "/Users/chongj/Desktop/Li_Lab/Projects/HEU/2026-05-21_MS1/CSM_annotations/"
os.makedirs(output_dir, exist_ok=True)

['RP_neg_final_table.tsv', 'HILIC_neg_final_table.tsv', 'HILIC_pos_final_table.tsv', 'RP_pos_final_table.tsv']


In [58]:
for f in ft_tables:
    # first features to empirical compounds
    this_file = ft_tbl_path + f
    print(this_file)

    mode = f.split("_final_table")[0].replace("_", "").lower()
    print(mode)

    extra_anno_dict = get_asari_default_anno(this_file)

    if "neg" in f:
        ms_mode = "neg"
        adduct_patterns = adduct_search_patterns_neg
        primary_ions_o = primary_ions_neg_ordered
    else:
        ms_mode = "pos"
        adduct_patterns = adduct_search_patterns
        primary_ions_o = primary_ions_pos_ordered

    anno, header, primary_fids, stats = annotate_asari_table_by_csm(
        this_file,
        KCD_formula_coordinate,
        isotope_search_patterns,
        adduct_patterns,
        extended_adducts,
        mDict,
        csmlib_dict[mode],
        neuCSM,
        primary_ions_ordered=primary_ions_o,
        mode=ms_mode,
        method=mode,
        mz_tolerance_ppm=5,
        rt_tolerance=2,
        quality_filter=False,
    )

    print(stats)

    write_csm_anno_2files(
        anno,
        header,
        extra_anno_dict,
        primary_fids,
        output_dir,
        f"{date.today().strftime('%Y-%m-%d')}_{mode}_CSM_annotation.tsv",
    )

/Volumes/T7/Li Lab/Projects/HEU/OLD/2025-12-14_HEU_Metabolomics_V3/heu_data/12_03_2025_final_feature_tables/RP_neg_final_table.tsv
rpneg
table header looks like: 
   ['id_number', 'mz', 'rtime', 'rtime_left_base', 'rtime_right_base', 'parent_masstrack_id', 'peak_area', 'cSelectivity', 'goodness_fitting', 'snr', 'detection_counts', 'Batch2_HEU011_B_RP_neg.mzML', 'Batch2_HEU015_M_RP_neg.mzML', 'Batch2_HEU026_B_RP_neg.mzML', 'Batch2_HEU027_B_RP_neg.mzML', 'Batch2_HEU028_B_RP_neg.mzML', 'Batch2_HEU029_B_RP_neg.mzML', 'Batch2_HEU032_M_RP_neg.mzML', 'Batch2_HEU034_B_RP_neg.mzML', 'Batch2_HEU034_M_RP_neg.mzML']
Read 5096 feature lines
Mass accuracy was estimated on 4346 matched values as -0.6 ppm.
No mz correction is needed.


Multiple charges considered: [1, 2, 3]


Khipu search grid: 
               M-H-       Na/H        K/H        ACN     NaCOOH  NaCH2COOH
M0        -1.007276  20.974664  36.948606  40.019273  66.980148  80.995724
13C/12C   -0.003921  21.978019  37.951961  41.022628  67.98